# 02 — Eksperimen Klasik (Skenario 1–10)

SVM & Random Forest dengan feature engineering manual. Jalankan **berurutan** setelah `01_eda.ipynb`.

In [ ]:
# ============================================================
# Setup cell - Kaggle Notebooks (Kaggle-only). Jalankan PALING ATAS.
# Cara attach dataset: panel kanan > + Add Data > cari
#   'fruit and vegetable disease healthy vs rotten' > Add.
# ============================================================
import os
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Clone repo dari GitHub (atau pull jika sudah ada di sesi ini)
REPO_URL = "https://github.com/faizhuda/pcd-kelompok-17.git"
PROJECT_DIR = Path("/kaggle/working/pcd-kelompok-17")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

# 2. Working directory ke root project + tambah ke sys.path
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 3. Dependency inti SUDAH pre-installed di Kaggle -> tidak ada pip install.

# 4. Dataset gambar (read-only, hasil + Add Data)
RAW_DATA_DIR = Path("/kaggle/input/fruit-and-vegetable-disease-healthy-vs-rotten")
assert RAW_DATA_DIR.exists(), "Attach dataset: + Add Data > cari nama dataset > Add."

# 5. Auto-restore hasil notebook sebelumnya (untuk notebook 03 & 04).
#    Attach output run lama via: + Add Data > Your Work / Dataset bersama.
def restore_previous_outputs():
    restored = []
    for inp in Path("/kaggle/input").glob("*"):
        repo = inp / "pcd-kelompok-17"
        if not repo.is_dir():
            continue
        for sub in ("results", "data/processed"):
            src_dir = repo / sub
            if src_dir.exists():
                shutil.copytree(src_dir, PROJECT_DIR / sub, dirs_exist_ok=True)
                restored.append(f'{inp.name}/{sub}')
    return restored

restored = restore_previous_outputs()
print("Project :", PROJECT_DIR)
print("Dataset :", RAW_DATA_DIR)
print("Restore :", restored or "(mulai dari nol)")


In [ ]:
import os
import sys
from pathlib import Path

# Setup cell sudah chdir ke PROJECT_DIR & menambah sys.path (Kaggle-only).
ROOT = Path("/kaggle/working/pcd-kelompok-17")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experiments import (
    run_classical_scenario,
    run_mcnemar_pair,
    select_best_enhancement,
)
from src.utils import build_dataset_index, get_project_paths, make_splits, set_seed

set_seed(42)
paths = get_project_paths()
# Split di-regenerate dari dataset (deterministik, SEED=42) - tidak baca splits.json
RAW_DATA_DIR = Path("/kaggle/input/fruit-and-vegetable-disease-healthy-vs-rotten")
train, val, test = make_splits(build_dataset_index(RAW_DATA_DIR))
cache_dir = paths["data_processed"]
metrics_dir = paths["metrics"]
figures_dir = paths["figures_confusion"]
models_dir = paths["models"]
print(len(train), len(val), len(test))


## Skenario 1–4: Perbandingan Enhancement

In [ ]:
val_f1_map = {}
scenario_results = {}

for sid in range(1, 5):
    print(f"\n=== Skenario {sid} ===")
    res = run_classical_scenario(
        sid, train, val, test,
        metrics_dir, figures_dir, models_dir, cache_dir,
    )
    val_f1_map[res["enhancement"]] = res["val_f1"]
    scenario_results[sid] = res
    print(f"Val F1: {res['val_f1']:.4f} | Test F1: {res['test_metrics']['f1_weighted']:.4f}")

best_enh = select_best_enhancement(val_f1_map, metrics_dir)
print(f"\nE* (enhancement terbaik): {best_enh}")


## Skenario 5–10: Segmentasi, Integrasi, Ablasi, RF

In [ ]:
for sid in range(5, 11):
    print(f"\n=== Skenario {sid} ===")
    res = run_classical_scenario(
        sid, train, val, test,
        metrics_dir, figures_dir, models_dir, cache_dir,
    )
    scenario_results[sid] = res
    print(f"Test F1: {res['test_metrics']['f1_weighted']:.4f}")


## Uji Signifikansi McNemar

In [ ]:
from src.utils import read_best_enhancement

best_enh = read_best_enhancement(metrics_dir)
best_sid = {"none": 1, "clahe": 2, "histeq": 3, "gamma": 4}[best_enh]
y_true = scenario_results[1]["y_test"]

run_mcnemar_pair(
    f"E*({best_enh}) vs S1",
    f"S{best_sid}", "S1",
    y_true,
    scenario_results[best_sid]["y_pred"],
    scenario_results[1]["y_pred"],
    metrics_dir,
)
run_mcnemar_pair(
    "S5 vs S1", "S5", "S1",
    y_true, scenario_results[5]["y_pred"], scenario_results[1]["y_pred"], metrics_dir,
)
run_mcnemar_pair(
    "S6 vs S1", "S6", "S1",
    y_true, scenario_results[6]["y_pred"], scenario_results[1]["y_pred"], metrics_dir,
)
print("McNemar (S11 vs S6) dijalankan di notebook 03.")


In [ ]:
import pandas as pd
sig_path = metrics_dir / "significance_tests.csv"
if sig_path.exists():
    display(pd.read_csv(sig_path))
